# 🚰 Notebook 2: Leaky Bucket

"Leaky bucket" names **two different things**, and mixing them up is the single most common rate-limiting mistake. Both appear in this notebook, clearly labelled:

| | **Shaper** (the real leaky bucket) | **Limiter** (leaky bucket *as a meter*) |
|---|---|---|
| What it does | **queues** requests and releases them at a fixed tempo | answers **yes/no** immediately |
| What the caller gets | a *delay*, then a response | an instant `200` or `429` |
| Output rate | constant, by construction | as bursty as whatever it admitted |
| Cost | added latency + a real queue to size | rejected requests |
| Used by | job dispatchers, ATM/Ethernet shaping, `nginx limit_req` w/ `delay` | GCRA, `redis-cell`, `nginx limit_req nodelay` |

Only the **shaper** actually smooths traffic. The limiter form is mathematically a token bucket with capacity 1 dressed differently — it *decides* at a steady rate, it doesn't *pace* anything.

### Mental picture 🎨
```
  incoming requests (bursty)
        │ │ │ │ │
        ▼ ▼ ▼ ▼ ▼
     ┌─────────────┐
     │ queue       │  capacity B (drop when full)
     └──────┬──────┘
            │  drains at leak_rate, no matter what
            ▼
        [ API ]
```


## 🛠️ Setup

```bash
cd 04-patterns/rate-limiting-and-throttling
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## ❌ Bad first try — unbounded queue

If you forget the capacity cap, a burst will buffer *forever*, using memory
and adding latency. Real servers OOM this way. Always bound the queue.


In [ ]:
import time
from collections import deque

class UnboundedLeakyBucket:
    """Bad: a shaper with no cap on the queue."""
    def __init__(self, leak_rate):
        self.leak_rate = leak_rate
        self.q = deque()

    def submit(self, req):
        self.q.append(req)     # no cap -> unbounded memory AND unbounded latency

bad = UnboundedLeakyBucket(leak_rate=5)
for i in range(100_000):
    bad.submit(i)

drain_seconds = len(bad.q) / bad.leak_rate
print(f'queued: {len(bad.q):,} requests')
print(f'time to drain at {bad.leak_rate}/s: {drain_seconds:,.0f}s ({drain_seconds/3600:.1f} hours)')
print('The last caller has been gone for hours. We are queueing work for nobody,')
print('holding the memory the whole time. An unbounded queue is not backpressure —')
print('it is a very expensive way to time out.')

## ✅ The shaper: bounded queue + paced release

The shaper answers a different question from every other limiter in this lab: not *"may I?"* but *"**when**?"*. Each admitted request is handed a **service time**, spaced exactly `1/leak_rate` apart. Requests that would have to wait past the queue's capacity are dropped.

In [ ]:
class ShapingLeakyBucket:
    """The real leaky bucket: schedules WHEN each request runs."""
    def __init__(self, leak_rate, capacity):
        self.interval = 1.0 / leak_rate   # spacing between departures
        self.capacity = capacity          # max requests waiting in the queue
        self.next_service = None          # when the next free slot opens

    def submit(self, t):
        """Returns the time this request will be SERVED, or None if the queue is full."""
        if self.next_service is None or self.next_service < t:
            self.next_service = t                       # queue is empty; serve now
        depth = (self.next_service - t) / self.interval  # how many are ahead of us
        if depth >= self.capacity:
            return None                                  # queue full -> drop
        served_at = self.next_service
        self.next_service += self.interval
        return served_at

# A tight burst of 8 arriving at once, leak_rate=5/s
lb = ShapingLeakyBucket(leak_rate=5, capacity=10)
for t in [0.0, 0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07]:
    s = lb.submit(t)
    print(f'arrived at {t:.2f}s -> served at {s:.2f}s (waited {s - t:.2f}s)')

## 📊 Shaper vs limiter, on the same traffic

Two bursts of 20, at `t=0` and `t=1.5`. We replay them through a token bucket (a limiter) and the shaper, and plot **when the downstream actually sees each request**.

Watch the y-axis rows: the limiter's output is as clumped as its input, just shorter. The shaper's output is a metronome.

In [ ]:
import matplotlib.pyplot as plt

class TokenBucket:      # a LIMITER, for contrast (notebook 1, with an injectable clock)
    def __init__(self, rate, capacity):
        self.rate, self.capacity = rate, capacity
        self.tokens, self.last = float(capacity), None
    def allow(self, t):
        if self.last is None:
            self.last = t
        self.tokens = min(self.capacity, self.tokens + (t - self.last) * self.rate)
        self.last = t
        if self.tokens >= 1:
            self.tokens -= 1
            return True
        return False

arrivals = [i * 0.005 for i in range(20)] + [1.5 + i * 0.005 for i in range(20)]

tb = TokenBucket(rate=5, capacity=10)
tb_out = [t for t in arrivals if tb.allow(t)]         # served at ARRIVAL time

sh = ShapingLeakyBucket(leak_rate=5, capacity=10)
pairs = [(t, sh.submit(t)) for t in arrivals]
sh_out = [s for _, s in pairs if s is not None]       # served at SCHEDULED time
sh_waits = [s - t for t, s in pairs if s is not None]

fig, ax = plt.subplots(figsize=(9, 2.6))
ax.eventplot(arrivals, colors='gray',  lineoffsets=2, linelengths=0.8)
ax.eventplot(tb_out,   colors='green', lineoffsets=1, linelengths=0.8)
ax.eventplot(sh_out,   colors='blue',  lineoffsets=0, linelengths=0.8)
ax.set_yticks([0, 1, 2])
ax.set_yticklabels(['shaper out', 'token out', 'arrivals'])
ax.set_xlabel('time (s)')
ax.set_title('The limiter admits bursts. Only the shaper paces them.')
plt.tight_layout(); plt.show()

print(f'arrivals           : {len(arrivals)}')
print(f'token bucket passed: {len(tb_out)}  (in 2 clumps, mirroring the input)')
print(f'shaper served      : {len(sh_out)}  (evenly spaced 0.20s apart)')
print(f'shaper dropped     : {sum(1 for _, s in pairs if s is None)}  (queue was full)')
print(f'shaper worst wait  : {max(sh_waits):.2f}s  <- the price of smoothing')

## 🧮 GCRA — the limiter form, in one float

**GCRA** (Generic Cell Rate Algorithm) is the leaky-bucket *limiter* squeezed into a single number: the **theoretical arrival time** (TAT) of the next conforming request. No queue, no counter, no timestamp list.

**Why people pick it**
- **O(1) state per key** — one float. 8 bytes in Redis vs. a sorted set of timestamps.
- Exact, with no window boundary to exploit.
- Used by the Redis [`redis-cell`](https://github.com/brandur/redis-cell) module, Shopify, and many CDNs.

> ⚠️ GCRA is **admission control** — it returns yes/no, and (usefully) *how long to wait*. It does not smooth output. If you need the downstream to see a constant tempo, you need the queue.

In [ ]:
class GCRA:
    """One-timestamp leaky-bucket limiter.
    rate  = sustained requests/sec
    burst = how many requests may arrive back-to-back from an idle state
    """
    def __init__(self, rate, burst):
        self.interval = 1.0 / rate               # ideal spacing between requests
        # tolerance is measured in seconds of "earliness" we forgive.
        # burst=1 means strict pacing, so the tolerance is (burst - 1) intervals.
        self.tolerance = (burst - 1) * self.interval
        self.tat = None                          # theoretical arrival time

    def allow(self, now=None):
        now = time.monotonic() if now is None else now
        tat = max(self.tat if self.tat is not None else now, now)
        if tat - now > self.tolerance:
            return False                         # too early -> reject
        self.tat = tat + self.interval
        return True

    def retry_after(self, now=None):
        """Seconds until the next request would conform — send this in Retry-After."""
        now = time.monotonic() if now is None else now
        if self.tat is None:
            return 0.0
        return max(0.0, (self.tat - self.tolerance) - now)

g = GCRA(rate=5, burst=10)                       # 5/s sustained, 10 back-to-back
print('burst of 15 from idle:', sum(g.allow() for _ in range(15)), 'allowed (expected 10)')
print(f'Retry-After          : {g.retry_after():.2f}s')
time.sleep(1.0)
print('after 1s of silence  :', sum(g.allow() for _ in range(8)), 'allowed (5 refilled)')

## 🧠 Quick reference

| | Token bucket (limiter) | GCRA (limiter) | Leaky bucket (shaper) |
|---|---|---|---|
| Answer | yes / no | yes / no + `Retry-After` | *when* |
| Allows bursts? | yes, up to bucket size | yes, up to `burst` | no — output is paced |
| Rejects when? | bucket empty | request is too early | queue full |
| Output rate | matches arrivals up to the burst | same | constant `leak_rate` |
| State per key | 2 floats | 1 float | 1 float + queue |
| Cost to the caller | a `429` | a `429` | added latency |

### 🧭 When to use which

- **Shaper** — when the *downstream* cannot take bursts: a legacy API with a hard concurrency limit, a payment processor with a contractual TPS, a database you're batch-writing to. You accept the latency because the alternative is breaking the downstream.
- **GCRA / token bucket** — when the *caller* is the one who must adapt. Public APIs: reject fast, tell them when to come back, keep your own memory flat.
- **Neither** — if the request is worth queueing for minutes, that's not a rate limiter, that's a **job queue** with a worker pool. Use one; you get durability and retries for free.

### 🚫 When NOT to shape

- **Interactive requests.** A user staring at a spinner does not want to be paced; they want a `429` and a retry, or a degraded answer now.
- **When you can't bound the queue.** An unbounded shaper is the failure mode from the first cell.
- **When arrival rate exceeds leak rate for long.** Shaping only absorbs *bursts*. If the average input is above the leak rate the queue fills and you're dropping anyway — with added latency on top. Size the queue for the burst you actually expect, then measure the wait.

### Real-world examples
- **`nginx limit_req`** — the shaper with `delay`/`burst=`, the limiter with `nodelay`. Same directive, both behaviours.
- **Network routers** — ATM/Ethernet traffic shaping (where GCRA was invented).
- **Background job dispatchers** — pacing writes into a rate-limited third-party API.